In [1]:
import pandas as pd
import numpy as np
import random
import copy
import math
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

In [2]:
file_name="SALEMOR_KINGElvis (6).xlsx"

ke_df=pd.read_excel(file_name,sheet_name='Elvis_Review')

wkend_overall_df=pd.read_excel('SALEMOR_CR.xlsx',sheet_name='WkEND-Overall')
wkend_route_df=pd.read_excel('SALEMOR_CR.xlsx',sheet_name='WkEND-RouteTotal')

df=pd.read_csv("elvissalemor2024obweekday_export_odbc(new_generation).csv")

wkday_overall_df=pd.read_excel('SALEMOR_CR.xlsx',sheet_name='WkDAY-Overall')
wkday_route_df=pd.read_excel('SALEMOR_CR.xlsx',sheet_name='WkDAY-RouteTotal')

ke_df=ke_df[ke_df['INTERV_INIT']!='999']
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['1st Cleaner']!='Test/No 5 MIN']
ke_df=ke_df[ke_df['Final_Usage'].str.lower()=='use']

In [3]:
df=pd.merge(df,ke_df['id'],on='id',how='inner')
df.drop_duplicates(subset='id',inplace=True)

def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [4]:
date_columns_check=['completed','datestarted']
date_columns=check_all_characters_present(df,date_columns_check)
date_columns

['Completed', 'Date_started']

In [5]:
def determine_date(row):
    if not pd.isnull(row[date_columns[0]]):
        return row[date_columns[0]]
    elif not pd.isnull(row[date_columns[1]]):
        return row[date_columns[1]]
    else:
        return pd.NaT

In [6]:
df['Date'] = df.apply(determine_date, axis=1)

In [7]:
date_string = df['Date'][0]

In [8]:
date_object = datetime.strptime(date_string, '%Y-%m-%d %H:%M:%S')

In [9]:
day_name = date_object.strftime('%A')
day_name

'Tuesday'

In [10]:
def get_day_name(x):
    date_object = datetime.strptime(x, '%Y-%m-%d %H:%M:%S')
    day_name = date_object.strftime('%A')
    return day_name

In [11]:
df['Day']=df['Date'].apply(get_day_name)


In [12]:
df[~df['Day'].isin(['Saturday','Sunday'])]

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,EXP_HOMELESSCode,EXP_HOMELESS,_REVERSE_TRIP,GENERATABLE_TRIPS,Date,Day
0,11584,2024-03-26 07:26:38,80,en,2024-03-26 07:14:39,2024-03-26 07:26:38,174.247.181.10,https://salem-or.etc-research.com/,-25200,MMH,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-26 07:26:38,Tuesday
1,11586,2024-03-26 07:43:33,80,en,2024-03-26 07:16:25,2024-04-01 23:49:51,174.247.186.8,https://salem-or.etc-research.com/,-25200,PBW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-26 07:43:33,Tuesday
2,11598,2024-03-26 07:29:19,80,en,2024-03-26 07:17:50,2024-05-20 10:14:33,172.56.152.18,https://salem-or.etc-research.com/,-25200,MDE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,2024-03-26 07:29:19,Tuesday
3,11600,2024-03-26 07:37:05,80,en,2024-03-26 07:24:59,2024-05-20 23:41:49,174.247.181.185,https://salem-or.etc-research.com/index.php/su...,-25200,BLC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,2024-03-26 07:37:05,Tuesday
4,11601,2024-03-26 07:48:59,80,en,2024-03-26 07:25:41,2024-05-20 23:42:16,172.56.153.15,https://salem-or.etc-research.com/,-25200,JLK,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reverse,2024-03-26 07:48:59,Tuesday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1683,15990,2024-04-18 18:04:49,80,en,2024-04-18 17:57:16,2024-04-18 18:04:49,174.247.190.222,https://salem-or.etc-research.com/,-25200,MDE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-18 18:04:49,Thursday
1684,15991,2024-04-18 18:09:00,80,en,2024-04-18 18:04:53,2024-04-18 18:09:00,174.247.190.222,https://salem-or.etc-research.com/index.php/su...,-25200,MDE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-18 18:09:00,Thursday
1685,15992,2024-04-18 18:33:37,80,en,2024-04-18 18:09:06,2024-04-18 18:33:37,174.247.190.222,https://salem-or.etc-research.com/index.php/su...,-25200,MDE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-18 18:33:37,Thursday
1686,15998,2024-04-18 20:00:49,80,en,2024-04-18 19:56:03,2024-04-18 20:00:49,174.247.190.222,https://salem-or.etc-research.com/index.php/su...,-25200,MDE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-04-18 20:00:49,Thursday


In [13]:
df.to_csv('Day Time SantaClarita.csv',index=False)